# ACT 模型训练 Notebook

使用此 Notebook 训练 ACT（Action Chunking with Transformers）模型。

**运行方式**: 按顺序执行每个 Cell，可在「配置参数」Cell 中调整超参数后重新训练。

## 1. 导入依赖

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader

# 确保项目根目录在 sys.path 中
PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from act.configuration_act import ACTConfig
from act.defaults import build_act_config, act_config_to_dict
from act.modeling_act import ACTModel
from act.ACTDataset import ACTDataset

print(f"PyTorch {torch.__version__}  |  Project root: {PROJECT_ROOT}")

## 2. 配置参数

修改以下参数以适配你的数据集和训练需求。

In [ ]:
# ============================================================
# 训练参数（按需修改）
# ============================================================

# 数据集目录（包含 data/ 和 meta/stats.json）
DATA_DIR = "proj57_database&model/dataset"

# 模型输出目录
OUTPUT_DIR = "checkpoints"

# 训练超参数
EPOCHS = 50               # 训练轮数
BATCH_SIZE = 8             # 批次大小
LR = 1e-4                  # 学习率
ACTION_CHUNK_SIZE = 8      # 动作分块大小
HIDDEN_DIM = 512            # 隐藏层维度

print(f"数据目录:     {DATA_DIR}")
print(f"输出目录:     {OUTPUT_DIR}")
print(f"训练轮数:     {EPOCHS}")
print(f"批次大小:     {BATCH_SIZE}")
print(f"学习率:       {LR}")
print(f"动作分块大小: {ACTION_CHUNK_SIZE}")
print(f"隐藏层维度:   {HIDDEN_DIM}")

## 3. 工具函数

数据集加载、归一化统计加载、设备选择。

In [ ]:
def get_device() -> str:
    """自动选择可用设备。"""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_dataset(data_dir: str) -> dict[str, torch.Tensor]:
    """加载 LeRobot 格式数据集，返回 image/state/action 张量。"""
    data_path = Path(data_dir)

    parquet_files = sorted(data_path.glob("data/chunk-*/file-*.parquet"))
    print(f"找到 {len(parquet_files)} 个数据文件")
    if not parquet_files:
        raise FileNotFoundError(f"未在 {data_path}/data/ 下找到 parquet 文件")

    images, states, actions = [], [], []

    for pf in parquet_files:
        df = pd.read_parquet(pf)

        for img_path in df["observation.image"]:
            full_path = data_path / img_path
            if full_path.exists():
                img = Image.open(full_path).convert("RGB")
                img = img.resize((224, 224))
                img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
                images.append(img_tensor)
            else:
                images.append(torch.zeros(3, 224, 224))

        for s in df["observation.state"]:
            states.append(torch.tensor(s, dtype=torch.float32))

        for a in df["action"]:
            actions.append(torch.tensor(a, dtype=torch.float32))

    images = torch.stack(images)
    states = torch.stack(states)
    actions = torch.stack(actions)

    print(f"加载完成: {len(images)} 样本, image={tuple(images.shape)}, state={tuple(states.shape)}, action={tuple(actions.shape)}")

    return {
        "observation.image": images,
        "observation.state": states,
        "action": actions,
    }


def load_norm_stats(data_dir: str) -> tuple[Optional[torch.Tensor], ...]:
    """从 stats.json 加载 QUANTILES 归一化参数。"""
    stats_path = Path(data_dir) / "meta" / "stats.json"
    if not stats_path.exists():
        print(f"⚠ stats.json 不存在 ({stats_path})，将不使用归一化")
        return None, None, None, None

    with open(stats_path) as f:
        stats = json.load(f)

    state_q01 = torch.tensor(stats["observation.state"]["q01"], dtype=torch.float32)
    state_q99 = torch.tensor(stats["observation.state"]["q99"], dtype=torch.float32)
    action_q01 = torch.tensor(stats["action"]["q01"], dtype=torch.float32)
    action_q99 = torch.tensor(stats["action"]["q99"], dtype=torch.float32)

    print(f"状态 q01={state_q01.tolist()} q99={state_q99.tolist()}")
    print(f"动作 q01={action_q01.tolist()} q99={action_q99.tolist()}")
    return state_q01, state_q99, action_q01, action_q99

## 4. 加载数据

In [ ]:
DEVICE = get_device()
print(f"使用设备: {DEVICE}")

# 加载数据集
data = load_dataset(DATA_DIR)

# 加载归一化统计
state_q01, state_q99, action_q01, action_q99 = load_norm_stats(DATA_DIR)

# 从数据推断维度
state_dim = data["observation.state"].shape[1]
action_dim = data["action"].shape[1]
print(f"数据维度: state_dim={state_dim}, action_dim={action_dim}")

## 5. 创建模型和 DataLoader

In [ ]:
# 构建配置
config = build_act_config(
    state_dim=state_dim,
    action_dim=action_dim,
    action_chunk_size=ACTION_CHUNK_SIZE,
    hidden_dim=HIDDEN_DIM,
)

# 创建模型
model = ACTModel(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params:,}")
print(f"配置: state_dim={config.state_dim}, action_dim={config.action_dim}, "
      f"chunk_size={config.action_chunk_size}, hidden_dim={config.hidden_dim}, "
      f"cvae={config.use_cvae}, temporal_ensembling={config.use_temporal_ensembling}")

# 创建数据集和 DataLoader
dataset = ACTDataset(
    data,
    action_chunk_size=ACTION_CHUNK_SIZE,
    normalize_images=True,
    state_q01=state_q01,
    state_q99=state_q99,
    action_q01=action_q01,
    action_q99=action_q99,
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"数据集大小: {len(dataset)}, 批次数: {len(dataloader)}")

# 优化器 + 移动模型到设备
optimizer = optim.Adam(model.parameters(), lr=LR)
model = model.to(DEVICE)
model.train()

## 6. 训练循环

每个 epoch 输出 loss / L1 / KL（CVAE 模式下），每 10 个 epoch 自动保存检查点。

In [ ]:
# CVAE latent 统计收集
all_mu, all_log_sigma = [], []
latent_collection_epochs = min(5, max(1, EPOCHS // 2))

# 创建输出目录
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

# 用于日志记录的 loss 列表
loss_history = []

for epoch in range(EPOCHS):
    total_loss = total_l1 = total_kl = 0.0
    num_batches = 0

    for batch in dataloader:
        images = batch["observation"]["image"].to(DEVICE)
        states = batch["observation"]["state"].to(DEVICE)
        actions = batch["action"].to(DEVICE)

        optimizer.zero_grad()
        output = model(images, states, action_target=actions, infer_cvae=False)

        predicted = output["action"]
        kl_loss = output.get("kl_loss")
        mu = output.get("mu")
        log_sigma_x2 = output.get("log_sigma_x2")

        if config.use_cvae and mu is not None and log_sigma_x2 is not None:
            if epoch >= EPOCHS - latent_collection_epochs:
                all_mu.append(mu.detach().cpu())
                all_log_sigma.append(log_sigma_x2.detach().cpu())

        l1_loss = F.l1_loss(predicted, actions)
        loss = l1_loss + (kl_loss * config.kl_weight) if kl_loss is not None else l1_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_l1 += l1_loss.item()
        if kl_loss is not None:
            total_kl += kl_loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    avg_l1 = total_l1 / num_batches
    avg_kl = total_kl / num_batches if total_kl > 0 else None
    loss_history.append((avg_loss, avg_l1, avg_kl))

    if config.use_cvae:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS}  loss={avg_loss:.6f}  L1={avg_l1:.6f}  KL={avg_kl:.6f}")
    else:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS}  loss={avg_loss:.6f}  L1={avg_l1:.6f}")

    # 每 10 个 epoch 保存检查点
    if (epoch + 1) % 10 == 0:
        ckpt_path = output_path / f"checkpoint_epoch_{epoch + 1}.pt"
        torch.save(model.state_dict(), ckpt_path)
        print(f"  保存检查点: {ckpt_path}")

print("\n训练完成!")

## 7. 训练 Loss 曲线

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(loss_history) + 1)

fig, axes = plt.subplots(1, 2 if config.use_cvae else 1, figsize=(8 if config.use_cvae else 5, 2.5), dpi=100)

if config.use_cvae:
    ax1, ax2 = axes
else:
    ax1 = axes

ax1.plot(epochs_range, [h[1] for h in loss_history], label="L1 Loss", marker="o", markersize=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("L1 Loss")
ax1.set_title("Training L1 Loss")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

if config.use_cvae:
    ax2.plot(epochs_range, [h[2] for h in loss_history], label="KL Loss", color="orange", marker="o", markersize=2)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("KL Loss")
    ax2.set_title("Training KL Loss")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 8. 保存最终模型

In [ ]:
final_path = output_path / "final_model.pt"

if config.use_cvae and len(all_mu) > 0:
    all_mu_tensor = torch.cat(all_mu, dim=0)
    all_log_sigma_tensor = torch.cat(all_log_sigma, dim=0)
    latent_mu_mean = all_mu_tensor.mean(dim=0)
    latent_log_sigma_mean = all_log_sigma_tensor.mean(dim=0)
    print(f"CVAE latent: mu={latent_mu_mean.mean().item():.4f}, log_sigma={latent_log_sigma_mean.mean().item():.4f}")

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "inference_latent_mu": latent_mu_mean,
        "inference_latent_log_sigma": latent_log_sigma_mean,
        "config": act_config_to_dict(config),
    }
    torch.save(checkpoint, final_path)
else:
    torch.save(model.state_dict(), final_path)

print(f"模型已保存到: {final_path}")
print(f"文件大小: {final_path.stat().st_size / 1024 / 1024:.1f} MB")